In [ ]:
# Quantum simulation libraries
from qutip import (
    basis, 
    mesolve, 
    qeye, 
    sigmax, 
    sigmay, 
    sigmaz, 
    tensor,
    )
import qutip
from qutip.measurement import measurement_statistics_observable

# Machine learning libraries
import torch
from torch import nn
from torch.utils.data import DataLoader, Dataset
import torch.nn.functional as F

# Plotting libraries
import matplotlib.pyplot as plt
from rich.progress import track

# Linalg libraries
import numpy as np
import h5py as hf

# Helper libraries
from dataclasses import dataclass

In [ ]:
# set the device
device = (
    "cuda"
    if torch.cuda.is_available()
    else "mps"
    if torch.backends.mps.is_available()
    else "cpu"
)
print(f"Using {device} device")

# Load target data

In [ ]:
background = np.genfromtxt("data/background.csv", delimiter=",", skip_header=1)[:, 1:][:500]
signal = np.genfromtxt("data/signal.csv", delimiter=",", skip_header=1)[:, 1:][:500]

In [ ]:
background = (background - background.mean(axis=0)) / background.std(axis=0)
signal = (signal - signal.mean(axis=0)) / signal.std(axis=0)

In [ ]:
class BFieldGenerator:
    def __init__(
        self,  raw_data: np.ndarray, times: np.ndarray
        ):
        """
        Create the BField generator.

        Parameters
        ----------
        amplitude : float
            Amplitude of the magnetic field.
        frequency : float
            Frequency of the cosine wave.
        resoltion : float
            Distance between measured points.
        length : int
            Amount of time to run it for.
        """
        self.raw_data = raw_data

        self.counter = 0

        self.measured_field = [self.raw_data[0]]
        self.measured_times = []
        self.times = times

    def __call__(self, t: float):
        """
        Get the next value of the magnetic field.

        Parameters
        ----------
        t : float
            Current time.
        """
        if t == self.times[self.counter + 1]:
            driving_field = self.raw_data[self.counter]

            self.measured_field.append(driving_field)      
            self.measured_times.append(t)
            self.counter += 1
        else:
            driving_field = self.measured_field[-1]

        return np.array(driving_field)

# Run Simulation

In [ ]:
@dataclass
class SimulationParameters:
    """
    Helper class for the simulation parameters.
    """
    length: int
    coupling: list

@dataclass
class SimulationState:
    """ 
    Helper class for the simulation state.
    """
    number_of_spins: int
    quantum_state: list
    spin_list: list
    coupling_list: list

In [ ]:
def get_simulation_state(parameters: SimulationParameters):
    """
    Returns the initial state of the simulation.
    """
    # Get the initial wavefunction
    number_of_spins = parameters.length ** 2
    initial_state = [basis(2, 0)] * (number_of_spins)

    # Setup operators for individual qubits
    sx_list, sy_list, sz_list = [], [], []
    for i in range(number_of_spins):
        op_list = [qeye(2)] * number_of_spins
        op_list[i] = sigmax()
        sx_list.append(tensor(op_list))
        op_list[i] = sigmay()
        sy_list.append(tensor(op_list))
        op_list[i] = sigmaz()
        sz_list.append(tensor(op_list))

    # Setup the operators for the coupling
    Jx = parameters.coupling * np.ones(number_of_spins)
    Jy = parameters.coupling * np.ones(number_of_spins)
    Jz = parameters.coupling * np.ones(number_of_spins)    

    return SimulationState(
        number_of_spins=number_of_spins,
        quantum_state=tensor(initial_state),
        spin_list=[sx_list, sy_list, sz_list],
        coupling_list=[Jx, Jy, Jz],
    )

In [ ]:
def compute_hamiltonian(t, args):
    """
    Compute the Hamiltonian at time t.
    
    Parameters
    ----------
    t : float
        Current time.
    args : dict
        System parameters in the H computation.
    """
    sx_list = args["sx_list"]
    sy_list = args["sy_list"]
    sz_list = args["sz_list"]
    Jx = args["Jx"]
    Jy = args["Jy"]
    Jz = args["Jz"]
    driving_field = args["driving"]

    N = args["N"]
    length = np.sqrt(N).astype(int)
    
    # Hamiltonian - Energy splitting terms
    field = driving_field(t)
    H = 0
    for i in range(9):
        H -= field[0] * sz_list[i]
        H -= field[1] * sx_list[i]

    lattice_sites = []
    for x in range(length):
        for y in range(length):
            lattice_sites.append([x, y])

    lattice_sites = np.array(lattice_sites)
    neighbours = np.array([[1, 0], [-1, 0], [0, 1], [0, -1]]) # right, left, top, bottom
    box_l = np.array([length, length])

    # Interaction terms
    spin_coupling_term = 0
    for n, site in enumerate(lattice_sites):

        neighbours = site[None, :] + neighbours
        neighbours_folded = neighbours - np.floor(neighbours / box_l[None, :]) * box_l[None, :]
        neighbour_indices = neighbours_folded[:, 0] + length * neighbours_folded[:, 1]

        for neighbour in neighbour_indices:
            spin_coupling_term += -Jx[n] * sx_list[n] * sx_list[int(neighbour)]
            spin_coupling_term += -Jy[n] * sy_list[n] * sy_list[int(neighbour)]
            spin_coupling_term += -Jz[n] * sz_list[n] * sz_list[int(neighbour)]

    return H + spin_coupling_term  # account for double counting
    

In [ ]:
simulation_parameters = SimulationParameters(
    length=3,
    coupling=5.0 * np.pi,
)
simulation_state = get_simulation_state(simulation_parameters)


times = np.linspace(0, 10, signal.shape[0])

signal_field = BFieldGenerator(signal, times)

args = {
    "sx_list": simulation_state.spin_list[0],
    "sy_list": simulation_state.spin_list[1],
    "sz_list": simulation_state.spin_list[2],
    "Jx": simulation_state.coupling_list[0],
    "Jy": simulation_state.coupling_list[1],
    "Jz": simulation_state.coupling_list[2],
    "N": simulation_state.number_of_spins,
    "driving": signal_field
}

In [ ]:
signal_results = mesolve(compute_hamiltonian, simulation_state.quantum_state, times, [], [], args)

In [ ]:
measured_signal_field = np.array(signal_field.measured_field)

plt.plot(measured_signal_field[:, 0], label="Measured")
plt.show()

In [ ]:
simulation_parameters = SimulationParameters(
    length=3,
    coupling=5 * np.pi,
)
simulation_state = get_simulation_state(simulation_parameters)


times = np.linspace(0, 10, background.shape[0])

bg_field = BFieldGenerator(background, times)

args = {
    "sx_list": simulation_state.spin_list[0],
    "sy_list": simulation_state.spin_list[1],
    "sz_list": simulation_state.spin_list[2],
    "Jx": simulation_state.coupling_list[0],
    "Jy": simulation_state.coupling_list[1],
    "Jz": simulation_state.coupling_list[2],
    "N": simulation_state.number_of_spins,
    "driving": bg_field
}

In [ ]:
bg_results = mesolve(compute_hamiltonian, simulation_state.quantum_state, times, [], [], args)

In [ ]:
measured_bg_field = np.array(bg_field.measured_field)

# plt.plot(background[:, 0], label="Measured")
# plt.plot(background[:, 1], label="Measured")
plt.plot(measured_bg_field[:, 0], label="Measured")

plt.show()

## Measure State Descriptions

In [ ]:
state_dimension = 1000
measure_states = 9
measurements = []

for _ in range(state_dimension):
    non_gue_matrix = qutip.rand_herm(
        2**measure_states, 
        0.9, 
        dims=[[2] * measure_states, [2] * measure_states]
    )
    measurements.append(non_gue_matrix)

In [ ]:
# Compute observables
signal_observations = np.zeros((signal.shape[0], state_dimension))
states = signal_results.states

for t, state in enumerate(states):
    for o, operator in enumerate(measurements):
        measure_state = state
        signal_observations[t][o] = qutip.expect(measure_state * measure_state.dag(), operator)

In [ ]:
# Compute observables
bg_observations = np.zeros((background.shape[0], state_dimension))
states = bg_results.states

for t, state in enumerate(states):
    for o, operator in enumerate(measurements):
        measure_state = state
        bg_observations[t][o] = qutip.expect(measure_state * measure_state.dag(), operator)

# Fit Model

In [ ]:
class NeuralNetwork(nn.Module):
    
    def __init__(self, state_dimension: int, output_dimension: int):
        """
        Build the network.
        
        Parameters
        ----------
        state_dimension : int
                Dimension of the state representation.
                This is the input to the layer.
        output_dimension : int
                Dimension of the output being predicted.
        """
        super().__init__()
        
        self.linear_stack = nn.Sequential(
            nn.Linear(state_dimension, 128),
            nn.ReLU(),
            nn.Linear(128, output_dimension)
        )
    
    def forward(self, x):
        """
        Forward pass through the network.
        
        As we are doing reservoir computing, this is 
        simply a linear layer.
        """
        return self.linear_stack(x)

In [ ]:
class ReservoirDataset(Dataset):
    """
    Custom dataset for the training.
    """
    def __init__(
        self, 
        state_data: np.ndarray, 
        function_data: np.ndarray,
    ):
        """
        Constructor for the dataset.

        Parameters
        ----------
        state_data : np.ndarray
                State description data.
        function_data : np.ndarray
                Function data being fit.
                This will be the target data.
        prediction_length : int
                How far into the future you will predict.
        """
        self.state_data = torch.Tensor(state_data).to(device)
        self.function_data = torch.Tensor(function_data).to(device)
        
        self.norm_factor = 1 # max(abs(function_data.flatten()))
    
    def __len__(self):
        """
        Length of the dataset.
        """
        return int(
            len(self.function_data)
        )
    
    def __getitem__(self, idx: int):
        """
        Collect an item from the dataset.
        
        Parameters
        ----------
        idx : int
                Index of the state to take.
        """
        state = self.state_data[idx]
        target = self.function_data[idx] / self.norm_factor
        
        return state, target

In [ ]:
def train(dataloader, model, loss_fn, optimizer):
    model.train()
    for batch, (X, y) in enumerate(dataloader):
        X, y = X.to(device), y.to(device)
        # Compute prediction error
        pred = model(X)
        loss = loss_fn(pred, y)

        # Backpropagation
        loss.backward()
        optimizer.step()
        optimizer.zero_grad()

        if batch % 100 == 0:
            loss = loss.item()
            
    return loss

In [ ]:
def test(dataloader, model, loss_fn) -> np.ndarray:
    size = len(dataloader.dataset)
    num_batches = len(dataloader)
    model.eval()
    test_loss = 0
    with torch.no_grad():
        for X, y in dataloader:
            X, y = X.to(device), y.to(device)
            pred = model(X)
            test_loss += loss_fn(pred, y).item()
    test_loss /= num_batches
    
    return test_loss

In [ ]:
def train_model(dataset, test_ds, model = None):
    """
    Train a model on the current data.
    """    
    if model is None:
        model = NeuralNetwork(
            state_dimension=12,
            output_dimension=2
        ).to(device)

        model = model.type(torch.float32)

    # Use MSE loss
    loss_fn = nn.MSELoss()

    # Use ADAM optimizer
    optimizer = torch.optim.Adam(model.parameters(), lr=1e-5, weight_decay=1e-5)

    # Create the loader
    loader = DataLoader(dataset, batch_size=512, shuffle=False)
    test_loader = DataLoader(test_ds, batch_size=512, shuffle=False)
    
    # Train the network
    epochs = 50000
    train_losses = []
    test_losses = []

    for t in track(range(epochs)):
        loss = train(loader, model, loss_fn, optimizer)
        train_losses.append(loss)
        loss = test(test_loader, model, loss_fn)
        test_losses.append(loss)
    try:
        train_losses = [item.cpu().detach().numpy() for item in train_losses]
    except AttributeError:
        pass
    
    return train_losses, test_losses, model

In [ ]:
split_fraction = 0.5
train_indices = np.random.choice(
    np.arange(0, background.shape[0] - 1),
    size=int((background.shape[0] - 1) * split_fraction),
    replace=False
)
test_indices = np.setdiff1d(np.arange(0, background.shape[0] - 1), train_indices)

train_state_data = bg_observations[train_indices, :]
train_function_data = background[train_indices]

test_state_data = bg_observations[test_indices, :]
test_function_data = background[test_indices]

train_ds = ReservoirDataset(
    state_data=train_state_data,
    function_data=train_function_data,
)

test_ds = ReservoirDataset(
    state_data=test_state_data,
    function_data=test_function_data,
)

train_losses, test_losses, model = train_model(train_ds, test_ds, model=None)


In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(10, 5))

ax[0].plot(test_losses)
ax[1].plot(train_losses)

ax[1].set_yscale("log")
ax[0].set_yscale("log")

plt.show()

In [ ]:
signal_ds = ReservoirDataset(
    state_data=signal_observations,
    function_data=signal,
)

bg_ds = ReservoirDataset(
    state_data=bg_observations,
    function_data=background,
)

In [ ]:
background_predictions = model(torch.Tensor(bg_observations).to(device)).cpu().detach().numpy()
signal_predictions = model(torch.Tensor(signal_observations).to(device)).cpu().detach().numpy()

In [ ]:
fig, ax = plt.subplots(1, 2)

for i in range(2):
    ax[i].plot(background[:, i], background_predictions[:, i], 'r.')
    ax[i].plot(background[:, i], background[:, i], 'k--')

plt.show()